# Preprocess STRING

This notebook processes STRING human data and isolates the LINCS-relevant subgraph(s)

# Imports

In [26]:
import matplotlib.pyplot as plt
import numpy as np
import networkx as nx
import os
import pandas as pd
from pathlib import Path
import pickle

# Directories

In [2]:
# Define current directory
cwd = Path.cwd()
# Define DATA directory
DATA = cwd.parents[1]/'data'/'canada'

# Define INPUT directory
INPUT = DATA / 'input'
# Define STRING directory
STRING = INPUT / 'string'

# Define OUTPUT directory
OUTPUT = DATA / 'output'

# Functions

## General

In [3]:
def file_to_list(path):
    '''
    Converts a .txt file to a list
    '''

    with open(f'{path}', 'r', encoding = 'utf-8') as f:
        list_file = [line.strip() for line in f]
    
    return list_file

def list_to_file(path, data):
      '''
      Saves a list or set to a .txt file with no header.
      '''

      with open(path, 'w') as f:
            for item in sorted(data):
                  f.write(f'{item}\n')

def pickle_load(path: str, report: bool = False):
    '''
    Loads pickled data.
    '''

    with open(path, 'rb') as f:
        data = pickle.load(f)

    if report == True:
        if type(data) == nx.Graph:
            num_nodes = len(data.nodes)
            num_edges = len(data.edges)
            print('>> pickle_load')
            print(f'Pickled graph loaded w/ {num_nodes:,} nodes and {num_edges:,} edges')
            print()
        else:
            print('>> pickle_load')
            print(f'Pickled file loaded')
            print()

    return data

def pickle_save(path: str, data, report: bool = False):
    '''
    Pickles data.
    '''

    with open(path, 'wb') as f:
        pickle.dump(data, f)

    if report == True:
        if type(data) == nx.Graph:
            num_nodes = len(data.nodes)
            num_edges = len(data.edges)
            print('>> pickle_save')
            print(f'Graph w/ {num_nodes:,} nodes and {num_edges:,} edges pickled')
        else:
            print('>> pickle_save')
            print(f'Data pickled')
            print()

# STRING

## Data

### Edges

Loads the STRING edgelist for the human PPI network.

In [5]:
# Load data
df_edges = pd.read_csv(STRING / '9606.protein.physical.links.full.v12.0.txt.gz', compression = 'gzip', sep = ' ')
# Show data
df_edges.head()

,protein1,protein2,homology,experiments,experiments_transferred,database,database_transferred,textmining,textmining_transferred,combined_score
0,9606.ENSP00000000233,9606.ENSP00000257770,0,312,0,0,0,0,0,311
1,9606.ENSP00000000233,9606.ENSP00000226004,0,162,0,0,0,0,0,161
2,9606.ENSP00000000233,9606.ENSP00000434442,0,0,0,500,0,0,0,499
3,9606.ENSP00000000233,9606.ENSP00000262455,0,531,0,0,0,0,0,531
4,9606.ENSP00000000233,9606.ENSP00000303145,0,0,0,500,0,0,0,499


### Node Info

Loads the STRING node information for the human PPI network.

In [6]:
# Load data
df_info = pd.read_csv(STRING / '9606.protein.info.v12.0.txt.gz', compression = 'gzip', sep = '\t')
# Show data
df_info.head()

,#string_protein_id,preferred_name,protein_size,annotation
0,9606.ENSP00000000233,ARF5,180,ADP-ribosylation factor 5; GTP-binding protein...
1,9606.ENSP00000000412,M6PR,277,Cation-dependent mannose-6-phosphate receptor;...
2,9606.ENSP00000001008,FKBP4,459,"Peptidyl-prolyl cis-trans isomerase FKBP4, N-t..."
3,9606.ENSP00000001146,CYP26B1,512,Cytochrome P450 26B1; Involved in the metaboli...
4,9606.ENSP00000002125,NDUFAF7,441,"Protein arginine methyltransferase NDUFAF7, mi..."


### Graph Data

Combines edgelist and node info.

In [17]:
# Define variables
SOURCE_COLUMN = 'protein1'
TARGET_COLUMN = 'protein2'
SCORE_COLUMN = 'combined_score'
ID_COLUMN = '#string_protein_id'
TAXON_ID = '9606.'

In [ ]:
# Copy edgelist
df_graph = df_edges.copy()
# Get data
df_graph = df_graph[[SOURCE_COLUMN, TARGET_COLUMN, SCORE_COLUMN]]

# Rename columns for merge
df_info.rename(columns = {ID_COLUMN : SOURCE_COLUMN}, inplace = True)
# Merge
df_graph = pd.merge(df_graph, df_info[[SOURCE_COLUMN, 'preferred_name', 'annotation']], how = 'left', on = SOURCE_COLUMN)
# Rename columns for merge
df_info.rename(columns = {SOURCE_COLUMN : TARGET_COLUMN}, inplace = True)
# Merge
df_graph = pd.merge(df_graph, df_info[[TARGET_COLUMN, 'preferred_name', 'annotation']], how = 'left', on  = TARGET_COLUMN)
# Reset column names
df_info.rename(columns = {TARGET_COLUMN : ID_COLUMN}, inplace = True)

# Get old column names
old_names = df_graph.columns
# Define new names
new_names = ['source_id', 'target_id', 'weight', 'source', 'source_annot', 'target', 'target_annot']
# Rename columns
df_graph.rename(columns = dict(zip(old_names, new_names)), inplace = True)

# Remove taxon ID
for column in ['source_id', 'target_id']:
    df_graph[column] = df_graph[column].str.replace(TAXON_ID, '')

# Define column order
column_order = ['source', 'target', 'weight', 'source_id', 'target_id', 'source_annot', 'target_annot']
# Set columns
df_graph = df_graph[column_order]

# Save data
pickle_save(OUTPUT / 'df_graph.pkl')
# Show data
df_graph.head()

,source,target,weight,source_id,target_id,source_annot,target_annot
0,ARF5,NT5E,311,ENSP00000000233,ENSP00000257770,ADP-ribosylation factor 5; GTP-binding protein...,5'-nucleotidase; Hydrolyzes extracellular nucl...
1,ARF5,DUSP3,161,ENSP00000000233,ENSP00000226004,ADP-ribosylation factor 5; GTP-binding protein...,Dual specificity protein phosphatase 3; Shows ...
2,ARF5,ARFGAP2,499,ENSP00000000233,ENSP00000434442,ADP-ribosylation factor 5; GTP-binding protein...,ADP-ribosylation factor GTPase-activating prot...
3,ARF5,ERP44,531,ENSP00000000233,ENSP00000262455,ADP-ribosylation factor 5; GTP-binding protein...,Endoplasmic reticulum resident protein 44; Med...
4,ARF5,TMED10,499,ENSP00000000233,ENSP00000303145,ADP-ribosylation factor 5; GTP-binding protein...,Transmembrane emp24 domain-containing protein ...


## Base Graph

Generates base networkx graph object of full human PPI network.

In [20]:
# Define variables
SCORE_FILTER = 0

In [ ]:
# Load data
df_graph = pickle_load(OUTPUT / 'df_graph.pkl')
# Define graph columns
list_columns = ['source', 'target', 'weight']
# Generate graph object
graph_base = nx.from_pandas_edgelist(df_graph[list_columns], source = 'source', target = 'target', edge_attr = 'weight')

# Record nodes/edges
num_nodes = len(graph_base.nodes)
num_edges = len(graph_base.edges)

# Define edges under weight filter
list_remove = [(source, target) for source, target, weight in graph_base.edges(data = 'weight') if weight < SCORE_FILTER]
# Remove edges under weight filter
graph_base.remove_edges_from(list_remove)
# Remove unconnected nodes
graph_base.remove_nodes_from(list(nx.isolates(graph_base)))

# Record nodes/edges
num_nodes_filter = len(graph_base.nodes)
num_edges_filter = len(graph_base.edges)
# Calculate variables
percent_nodes = num_nodes_filter / num_nodes * 100
percent_edges = num_edges_filter / num_edges * 100

# Report
print(f'Base graph of {num_nodes:,} nodes and {num_edges:,} edges generated')
print(f'Filtered for edges with a confidence score >= {SCORE_FILTER}')
print(f'{percent_nodes:.2f}% nodes ({num_nodes_filter:,}/{num_nodes:,}) and {percent_edges:.2f}% edges ({num_edges_filter:,}/{num_edges:,}) remain')
print()

# Save graph
pickle_save(OUTPUT / 'graph_base.pkl', graph_base)

Base graph of 18,767 nodes and 738,805 edges generated
Filtered for edges with a confidence score >= 0
100.00% nodes (18,767/18,767) and 100.00% edges (738,805/738,805) remain



## Landmark Graph

Filters base graph using mapped landmark genes i.e. those with gene expression data.

In [ ]:
# Load base grah
graph_base = pickle_load(OUTPUT / 'graph_base.pkl')
# Load mapped landmark genes
list_landmark_mapped = file_to_list(OUTPUT / 'list_landmark_mapped.txt')

# Extract nodelist
list_nodes = list(graph_base.nodes)
# Get landmark nodes not in graph_base
list_missing = set(list_landmark_mapped) - set(list_nodes)
# Get landmark nodes in graph_base
list_found = set(list_landmark_mapped).intersection(set(list_nodes))

# Get nodes to remove
list_remove = set(list_nodes) - set(list_landmark_mapped)
# Copy graph
graph_landmark = graph_base.copy()
# Remove nodes
graph_landmark.remove_nodes_from(list_remove)
# Get isolated nodes
list_isolates = list(nx.isolates(graph_landmark))
# Remove isolated nodes
graph_landmark.remove_nodes_from(list_isolates)

# Report
len_landmark_mapped = len(list_landmark_mapped)
len_missing = len(list_missing)
len_overlap = len(list_found)
len_nodes = len(graph_base.nodes)
len_nodes_lm = len(graph_landmark.nodes)
len_isolates = len(list_isolates)
percent_missing = len_missing / len_landmark_mapped * 100 
percent_overlap = len_overlap / len_landmark_mapped * 100
percent_isolates = len_isolates / len_overlap * 100
percent_nodes = len_nodes_lm / len_nodes * 100
percent_landmark = len_nodes_lm / len_landmark_mapped * 100

print(f'{len_landmark_mapped:,} landmark genes identified')
print(f'{percent_overlap:.2f}% of landmark genes ({len_overlap:,}/{len_landmark_mapped:,}) found in graph_base')
print(f'{percent_isolates:.2f}% of landmark genes ({len_isolates:,}/{len_overlap:,}) in subgraph found as isolates')
print(f'{percent_nodes:.2f}% of nodes ({len_nodes_lm:,}/{len_nodes:,}) remain after filtering graph_base for landmark nodes and removing isolates')
print(f'{percent_landmark:.2f}% of landmark genes ({len_nodes_lm:,}/{len_landmark_mapped:,}) retained in graph_landmark')

# Save graph
pickle_save(OUTPUT / 'graph_landmark.pkl', graph_landmark)

945 landmark genes identified
99.79% of landmark genes (943/945) found in graph_base
2.97% of landmark genes (28/943) in subgraph found as isolates
4.88% of nodes (915/18,767) remain after filtering graph_base for landmark nodes and removing isolates
96.83% of landmark genes (915/945) retained in graph_landmark
